In [1]:
import os
os.chdir("/home/mdarif/MLOPS/Wine-Quality-Prediction")
print(os.getcwd())

/home/mdarif/MLOPS/Wine-Quality-Prediction


In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_file: Path
    test_file: Path
    model_file: Path
    status_file: Path
    kernel: str
    gamma: str
    epsilon: float
    C: float

In [3]:
from wine_quality_prediction.constants import *
from wine_quality_prediction.utils import read_yaml, create_directories

class ConfiguationManager:

    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH,
            schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.SVR
        create_directories([config.root_dir])
        return ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_file=Path(config.train_file),
            test_file=Path(config.test_file),
            model_file=Path(config.model_file),
            status_file=Path(config.status_file),
            kernel=params.kernel,
            gamma=params.gamma,
            epsilon=params.epsilon,
            C=params.C
        )

In [4]:
import pandas as pd
from sklearn.svm import SVR
from wine_quality_prediction.logger import get_logger, ModelTrainingError
from wine_quality_prediction.utils import save_bin, get_size, save_status

logger = get_logger("model_trainer", "model_trainer.log")

In [5]:
class ModelTrainer:
    """Handles model training using SVR."""

    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def load_data(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Load train and test data."""

        try:
            logger.info(f"Reading train_file: '{self.config.train_file}'")
            logger.info(f"Train file size: '{get_size(self.config.train_file)}'")

            logger.info(f"Reading test file: '{self.config.test_file}'")
            logger.info(f"Test file size: '{get_size(self.config.test_file)}'")

            train = pd.read_csv(self.config.train_file)
            test = pd.read_csv(self.config.test_file)

            logger.info(f"Train shape: {train.shape}")
            logger.info(f"Test shape: {test.shape}")

            return train, test

        except Exception as e:
            logger.error(
                f"Data load failed - "
                f"train='{self.config.train_file}' "
                f"test='{self.config.test_file}': {e}",
                exc_info=True
            )
            raise ModelTrainingError("Failed to load data") from e

    def train(self, train: pd.DataFrame) -> SVR:
        """Train SVR model."""

        try:
            target_column = "quality"

            x_train = train.drop(columns=[target_column])
            y_train = train[target_column]
            logger.info(
                f"Training SVR - "
                f"kernel='{self.config.kernel}' "
                f"gamma='{self.config.gamma}' "
                f"epsilon='{self.config.epsilon}' "
                f"C='{self.config.C}' "
            )
            model = SVR(
                kernel=self.config.kernel,
                gamma=self.config.gamma,
                epsilon=self.config.epsilon,
                C=self.config.C
            )
            model.fit(x_train, y_train)

            save_bin(
                model,
                self.config.model_file
            )

            logger.info(f"Model saved at: '{self.config.model_file}'")
            return model

        except Exception as e:
            logger.error(
                f"Model training failed: {e}",
                exc_info=True
            )
            raise ModelTrainingError("Failed to train model") from e

    def run(self):
        """Run full model training pipeline."""

        try:
            train, _ = self.load_data()
            self.train(train)
            save_status(
                self.config.status_file,
                "model_trainer",
                True
            )
            logger.info("Model training completed successfully")

        except Exception as e:
            save_status(
                self.config.status_file,
                "model_trainer",
                False
            )
            logger.error(
                f"Unexpected error: {e}",
                exc_info=True
            )
            raise ModelTrainingError("Model training pipeline failed") from e

In [6]:
try:
    config = ConfiguationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.run()
except Exception:
    raise

2026-06-02 08:17:35,743 - common - INFO - common.py:31 - YAML file loaded successfully: 'config/config.yaml'
2026-06-02 08:17:35,745 - common - INFO - common.py:31 - YAML file loaded successfully: 'params.yaml'
2026-06-02 08:17:35,748 - common - INFO - common.py:31 - YAML file loaded successfully: 'schema.yaml'
2026-06-02 08:17:35,749 - common - INFO - common.py:53 - Created directory at: 'artifacts'
2026-06-02 08:17:35,751 - common - INFO - common.py:53 - Created directory at: 'artifacts/model_trainer'
2026-06-02 08:17:35,752 - model_trainer - INFO - 4153399558.py:11 - Reading train_file: 'artifacts/data_transformation/train.csv'
2026-06-02 08:17:35,753 - model_trainer - INFO - 4153399558.py:12 - Train file size: '~196 KB'
2026-06-02 08:17:35,753 - model_trainer - INFO - 4153399558.py:14 - Reading test file: 'artifacts/data_transformation/test.csv'
2026-06-02 08:17:35,754 - model_trainer - INFO - 4153399558.py:15 - Test file size: '~49 KB'
2026-06-02 08:17:35,764 - model_trainer - INF

2026-06-02 08:17:35,811 - common - INFO - common.py:111 - Binary file saved at: 'artifacts/model_trainer/model.joblib'
2026-06-02 08:17:35,812 - model_trainer - INFO - 4153399558.py:62 - Model saved at: 'artifacts/model_trainer/model.joblib'
2026-06-02 08:17:35,813 - common - INFO - common.py:72 - JSON file saved at: 'artifacts/model_trainer/status.json'
2026-06-02 08:17:35,814 - common - INFO - common.py:152 - Status file saved at: 'artifacts/model_trainer/status.json' for stage: 'model_trainer' with status: 'True'
2026-06-02 08:17:35,814 - model_trainer - INFO - 4153399558.py:83 - Model training completed successfully
